# 📋 Hallazgos del EDA

## Resumen ejecutivo

1. Total de registros analizados

Se realizó un análisis exploratorio y de calidad sobre la tabla:

airline_catalog.bronze.flights_bronze

El dataset contiene:

Total de registros analizados: 539.747 vuelos

El período corresponde a enero de 2025 e incluye información operacional de vuelos: aerolínea, origen, destino, horarios programados y reales, tiempos de vuelo, distancia, cancelaciones y causas de demora.

2. Principales problemas de calidad encontrados

Durante el análisis se identificaron los siguientes puntos:

Valores nulos

Los principales campos con valores faltantes fueron:

CANCELLATION_CODE: 96,98% de valores nulos.
CARRIER_DELAY: 81,82% de valores nulos.
WEATHER_DELAY: 81,82% de valores nulos.
NAS_DELAY: 81,82% de valores nulos.
SECURITY_DELAY: 81,82% de valores nulos.

Estos valores no representan necesariamente errores, ya que dependen de eventos específicos del vuelo:

El código de cancelación solo existe cuando el vuelo fue cancelado.
Las causas de demora solo se completan cuando existe una demora atribuida a esa causa.

Problemas de formato

Se identificaron campos que requieren normalización:

FL_DATE almacenado como string.
Campos horarios (CRS_DEP_TIME, DEP_TIME, CRS_ARR_TIME, ARR_TIME) almacenados como enteros en formato HHMM.

Ejemplo:

1301 → 13:01

Duplicados y valores inválidos
Registros duplicados detectados: 0
Valores negativos en DEP_DELAY_NEW: 0
Distancias inválidas detectadas: 0

El dataset presenta consistencia en términos de unicidad y rangos básicos.

3. Porcentaje de datos válidos vs inválidos

Del total de registros analizados:

Datos válidos: aproximadamente 100% en términos de estructura, duplicados y valores críticos evaluados.
Datos con observaciones: asociados principalmente a valores nulos esperados por lógica de negocio.

Los valores faltantes encontrados corresponden principalmente a ausencia de eventos (cancelaciones o causas específicas de demora) y no a problemas de carga.

4. Recomendaciones para limpieza en Silver

Para la construcción de la capa Silver se recomienda:

Convertir FL_DATE a formato DATE.
Transformar horarios HHMM a formato TIME/TIMESTAMP.
Cambiar campos booleanos:
CANCELLED
DIVERTED
de valores numéricos a tipo boolean.
Mantener valores NULL en campos donde representan ausencia de eventos.
Validar relación entre vuelos cancelados y CANCELLATION_CODE.
Crear columnas derivadas:
estado del vuelo
duración del vuelo
diferencia entre horario programado y real
clasificación de demora.
Eliminar columnas técnicas como _rescued_data si no contiene información.
Conclusión

El dataset presenta una calidad adecuada para avanzar hacia la capa Silver. Los principales ajustes corresponden a normalización de tipos de datos y creación de reglas de negocio, mientras que no se detectaron problemas críticos de duplicidad o registros inválidos.

In [0]:
DESCRIBE airline_catalog.bronze.flights_bronze

In [0]:
--Total registros en la tabla flights_bronze
SELECT * FROM airline_catalog.bronze.flights_bronze
limit 5

In [0]:
--Creamos Una CTE para realizar el calculo de los valores nulos por cada columna
With base AS (
    SELECT * FROM airline_catalog.bronze.flights_bronze
), 

nulos AS (
 SELECT
 COUNT(*) AS TOTAL_REGISTROS,
  SUM(CASE WHEN DEP_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_DEP_TIME,
  SUM(CASE WHEN DEP_DELAY_NEW IS NULL THEN 1 ELSE 0 END) AS NULL_DEP_DELAY,
  SUM(CASE WHEN ARR_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_ARR_TIME,
  SUM(CASE WHEN ARR_DELAY_NEW IS NULL THEN 1 ELSE 0 END) AS NULL_ARR_DELAY,
  SUM(CASE WHEN CANCELLATION_CODE IS NULL THEN 1 ELSE 0 END) AS NULL_CANCELLATION_CODE,
  SUM(CASE WHEN ACTUAL_ELAPSED_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_ACTUAL_ELAPSED_TIME,
  SUM(CASE WHEN AIR_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_AIR_TIME,
  SUM(CASE WHEN CARRIER_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_CARRIER_DELAY,
  SUM(CASE WHEN WEATHER_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_WEATHER_DELAY,
  SUM(CASE WHEN NAS_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_NAS_DELAY,
  SUM(CASE WHEN SECURITY_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_SECURITY_DELAY
  FROM base
)
--Calculo de porcentaje de nulos por columna
SELECT
  TOTAL_REGISTROS,
  NULL_DEP_TIME,
  ROUND(NULL_DEP_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_DEP_TIME,
  NULL_DEP_DELAY,
  ROUND(NULL_DEP_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_DEP_DELAY,
  NULL_ARR_TIME,
  ROUND(NULL_ARR_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_ARR_TIME,
  NULL_ARR_DELAY,
  ROUND(NULL_ARR_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_ARR_DELAY,
  NULL_CANCELLATION_CODE,
  ROUND(NULL_CANCELLATION_CODE * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_CANCELLATION_CODE,
  NULL_ACTUAL_ELAPSED_TIME,
  ROUND(NULL_ACTUAL_ELAPSED_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_ACTUAL_ELAPSED_TIME,
  NULL_AIR_TIME,
  ROUND(NULL_AIR_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_AIR_TIME,
  NULL_CARRIER_DELAY,
  ROUND(NULL_CARRIER_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_CARRIER_DELAY,
  NULL_WEATHER_DELAY,
  ROUND(NULL_WEATHER_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_WEATHER_DELAY,
  NULL_NAS_DELAY,
  ROUND(NULL_NAS_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_NAS_DELAY,
  NULL_SECURITY_DELAY,
  ROUND(NULL_SECURITY_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_SECURITY_DELAY
FROM nulos;

In [0]:
--Cardinalidad por columna para saber cuantos valores unicos existen en c/u
SELECT
COUNT(DISTINCT YEAR) as anio,
COUNT(DISTINCT MONTH) as mes,
COUNT(DISTINCT DAY_OF_MONTH) as dia,
COUNT(DISTINCT FL_DATE) as fecha,
COUNT(DISTINCT OP_UNIQUE_CARRIER) as operador,
COUNT(DISTINCT TAIL_NUM) as avion,
COUNT(DISTINCT OP_CARRIER_FL_NUM) as vuelo,
COUNT(DISTINCT ORIGIN_AIRPORT_ID) as aeropuerto_origen,
COUNT(DISTINCT ORIGIN) as ciudad_origen,
COUNT(DISTINCT ORIGIN_CITY_NAME) as ciudad_origen_nombre,
COUNT(DISTINCT ORIGIN_STATE_NM) as estado_origen,
COUNT(DISTINCT DEST_AIRPORT_ID) as aeropuerto_destino,
COUNT(DISTINCT DEST) as ciudad_destino,
COUNT(DISTINCT DEST_CITY_NAME) as ciudad_destino_nombre,
COUNT(DISTINCT DEST_STATE_NM) as estado_destino,
COUNT(DISTINCT CRS_DEP_TIME) as hora_salida,
COUNT(DISTINCT DEP_TIME) as hora_salida_real,
COUNT(DISTINCT DEP_DELAY_NEW) as retraso_salida,
COUNT(DISTINCT CRS_ARR_TIME) as hora_llegada,
COUNT(DISTINCT ARR_TIME) as hora_llegada_real,
COUNT(DISTINCT ARR_DELAY_NEW) as retraso_llegada,
COUNT(DISTINCT CANCELLED) as vuelo_cancelado,
COUNT(DISTINCT CANCELLATION_CODE) as codigo_cancelacion,
COUNT(DISTINCT DIVERTED) as vuelo_divertido,
COUNT(DISTINCT ACTUAL_ELAPSED_TIME) as tiempo_vuelo,
COUNT(DISTINCT AIR_TIME) as tiempo_vuelo_real,
COUNT(DISTINCT DISTANCE) as distancia,
COUNT(DISTINCT CARRIER_DELAY) as retraso_operador,
COUNT(DISTINCT WEATHER_DELAY) as retraso_clima,
COUNT(DISTINCT NAS_DELAY) as retraso_nas,
COUNT(DISTINCT SECURITY_DELAY) as retraso_seguridad,
COUNT(DISTINCT LATE_AIRCRAFT_DELAY) as retraso_avion
FROM airline_catalog.bronze.flights_bronze


In [0]:
--distribucion de variables categoricas para ver como se reparten los datos
--hay una aereolinea dominante?

SELECT
 OP_UNIQUE_CARRIER,
 COUNT(*) AS vuelos,
 ROUND(COUNT(*) * 100.0 /
 (SELECT COUNT(*) FROM airline_catalog.bronze.flights_bronze),2) AS porcentaje
FROM airline_catalog.bronze.flights_bronze
GROUP BY OP_UNIQUE_CARRIER
ORDER BY vuelos DESC;

In [0]:
--Aeropuertos mas utilizados ORIGEN:
SELECT
 ORIGIN,
 COUNT(*) AS vuelos
FROM airline_catalog.bronze.flights_bronze
GROUP BY ORIGIN
ORDER BY vuelos DESC
LIMIT 10;

In [0]:
--Aeropuertos mas utilizados DESTINO:
SELECT
 DEST,
 COUNT(*) AS vuelos
FROM airline_catalog.bronze.flights_bronze
GROUP BY DEST
ORDER BY vuelos DESC
LIMIT 10;

In [0]:
--Estadisticas descripticas- Variables Numericas
SELECT
  'DEP_DELAY_NEW' AS columna,
  COUNT(DEP_DELAY_NEW) AS registros_validos,
  MIN(DEP_DELAY_NEW) AS minimo,
  MAX(DEP_DELAY_NEW) AS maximo,
  ROUND(AVG(DEP_DELAY_NEW),2) AS promedio,
  ROUND(STDDEV(DEP_DELAY_NEW),2) AS desviacion_estandar
FROM airline_catalog.bronze.flights_bronze

UNION ALL

SELECT
  'ARR_DELAY_NEW',
  COUNT(ARR_DELAY_NEW),
  MIN(ARR_DELAY_NEW),
  MAX(ARR_DELAY_NEW),
  ROUND(AVG(ARR_DELAY_NEW),2),
  ROUND(STDDEV(ARR_DELAY_NEW),2)
FROM airline_catalog.bronze.flights_bronze

UNION ALL

SELECT
  'DISTANCE',
  COUNT(DISTANCE),
  MIN(DISTANCE),
  MAX(DISTANCE),
  ROUND(AVG(DISTANCE),2),
  ROUND(STDDEV(DISTANCE),2)
FROM airline_catalog.bronze.flights_bronze

UNION ALL

SELECT
  'AIR_TIME',
  COUNT(AIR_TIME),
  MIN(AIR_TIME),
  MAX(AIR_TIME),
  ROUND(AVG(AIR_TIME),2),
  ROUND(STDDEV(AIR_TIME),2)
FROM airline_catalog.bronze.flights_bronze

UNION ALL

SELECT
  'ACTUAL_ELAPSED_TIME',
  COUNT(ACTUAL_ELAPSED_TIME),
  MIN(ACTUAL_ELAPSED_TIME),
  MAX(ACTUAL_ELAPSED_TIME),
  ROUND(AVG(ACTUAL_ELAPSED_TIME),2),
  ROUND(STDDEV(ACTUAL_ELAPSED_TIME),2)
FROM airline_catalog.bronze.flights_bronze;

In [0]:
--Reporte con: total de registros, nulos por columnas importantes, porcentaje de nulos, valores válidos / inválidos, duplicados
WITH base AS (

    SELECT *
    FROM airline_catalog.bronze.flights_bronze

),

metricas AS (

    SELECT

        COUNT(*) AS total_registros,

        -- Nulos
        SUM(CASE WHEN DEP_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_DEP_TIME,
        SUM(CASE WHEN DEP_DELAY_NEW IS NULL THEN 1 ELSE 0 END) AS NULL_DEP_DELAY,
        SUM(CASE WHEN ARR_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_ARR_TIME,
        SUM(CASE WHEN ARR_DELAY_NEW IS NULL THEN 1 ELSE 0 END) AS NULL_ARR_DELAY,
        SUM(CASE WHEN CANCELLATION_CODE IS NULL THEN 1 ELSE 0 END) AS NULL_CANCELLATION_CODE,
        SUM(CASE WHEN ACTUAL_ELAPSED_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_ACTUAL_ELAPSED_TIME,
        SUM(CASE WHEN AIR_TIME IS NULL THEN 1 ELSE 0 END) AS NULL_AIR_TIME,
        SUM(CASE WHEN CARRIER_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_CARRIER_DELAY,
        SUM(CASE WHEN WEATHER_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_WEATHER_DELAY,
        SUM(CASE WHEN NAS_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_NAS_DELAY,
        SUM(CASE WHEN SECURITY_DELAY IS NULL THEN 1 ELSE 0 END) AS NULL_SECURITY_DELAY,

        -- Valores placeholder / sospechosos
        SUM(CASE WHEN DEP_DELAY_NEW < 0 THEN 1 ELSE 0 END) AS dep_delay_negativos,
        SUM(CASE WHEN DISTANCE <= 0 THEN 1 ELSE 0 END) AS distance_invalidos,

        -- Cancelaciones
        SUM(CASE WHEN CANCELLED = 1 THEN 1 ELSE 0 END) AS vuelos_cancelados

    FROM base

)

SELECT

    total_registros,
 NULL_DEP_TIME,
  ROUND(NULL_DEP_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_DEP_TIME,
  NULL_DEP_DELAY,
  ROUND(NULL_DEP_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_DEP_DELAY,
  NULL_ARR_TIME,
  ROUND(NULL_ARR_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_ARR_TIME,
  NULL_ARR_DELAY,
  ROUND(NULL_ARR_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_ARR_DELAY,
  NULL_CANCELLATION_CODE,
  ROUND(NULL_CANCELLATION_CODE * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_CANCELLATION_CODE,
  NULL_ACTUAL_ELAPSED_TIME,
  ROUND(NULL_ACTUAL_ELAPSED_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_ACTUAL_ELAPSED_TIME,
  NULL_AIR_TIME,
  ROUND(NULL_AIR_TIME * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_AIR_TIME,
  NULL_CARRIER_DELAY,
  ROUND(NULL_CARRIER_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_CARRIER_DELAY,
  NULL_WEATHER_DELAY,
  ROUND(NULL_WEATHER_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_WEATHER_DELAY,
  NULL_NAS_DELAY,
  ROUND(NULL_NAS_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_NAS_DELAY,
  NULL_SECURITY_DELAY,
  ROUND(NULL_SECURITY_DELAY * 100.0 /  TOTAL_REGISTROS, 2) AS PCT_NULL_SECURITY_DELAY,
  dep_delay_negativos,
  distance_invalidos,
  vuelos_cancelados,
  ROUND(vuelos_cancelados * 100.0 / total_registros,2) AS pct_cancelados

FROM metricas;

In [0]:
-- Valores Duplicados
WITH duplicados AS (

    SELECT
        FL_DATE,
        OP_UNIQUE_CARRIER,
        OP_CARRIER_FL_NUM,
        ORIGIN,
        DEST,
        CRS_DEP_TIME,
        COUNT(*) AS cantidad

    FROM airline_catalog.bronze.flights_bronze

    GROUP BY
        FL_DATE,
        OP_UNIQUE_CARRIER,
        OP_CARRIER_FL_NUM,
        ORIGIN,
        DEST,
        CRS_DEP_TIME

    HAVING COUNT(*) > 1

)

SELECT *
FROM duplicados
ORDER BY cantidad DESC;

In [0]:
--¿Qué aerolíneas tienen más operaciones en enero 2025?
--Uso de Window Function 
WITH vuelos_por_aerolinea AS (

    SELECT
        OP_UNIQUE_CARRIER,
        COUNT(*) AS total_vuelos
    FROM airline_catalog.bronze.flights_bronze
    GROUP BY OP_UNIQUE_CARRIER

)

SELECT
    OP_UNIQUE_CARRIER,
    total_vuelos,
    RANK() OVER(
        ORDER BY total_vuelos DESC
    ) AS ranking

FROM vuelos_por_aerolinea;

In [0]:
--¿Cuáles son los aeropuertos más utilizados?
WITH aeropuertos AS (

    SELECT
        ORIGIN,
        COUNT(*) AS vuelos

    FROM airline_catalog.bronze.flights_bronze

    GROUP BY ORIGIN

)

SELECT
    ORIGIN,
    vuelos,

    RANK() OVER(
        ORDER BY vuelos DESC
    ) AS ranking

FROM aeropuertos;

In [0]:
--Ranking de rutas más frecuentes
WITH rutas AS (

SELECT
    ORIGIN,
    DEST,
    COUNT(*) AS cantidad_vuelos

FROM airline_catalog.bronze.flights_bronze

GROUP BY
    ORIGIN,
    DEST

)

SELECT
    ORIGIN,
    DEST,
    cantidad_vuelos,

    RANK() OVER(
        ORDER BY cantidad_vuelos DESC
    ) AS ranking

FROM rutas;

In [0]:
--Ranking de demoras por aerolínea
WITH delays AS (

SELECT
    OP_UNIQUE_CARRIER,
    ROUND(AVG(DEP_DELAY_NEW),2) AS promedio_delay

FROM airline_catalog.bronze.flights_bronze

GROUP BY OP_UNIQUE_CARRIER

)

SELECT
    OP_UNIQUE_CARRIER,
    promedio_delay,

    RANK() OVER(
        ORDER BY promedio_delay DESC
    ) AS ranking_delay

FROM delays;

In [0]:
--Comparar vuelo contra promedio de su aerolínea
SELECT

    OP_UNIQUE_CARRIER,
    OP_CARRIER_FL_NUM,
    DEP_DELAY_NEW,

    AVG(DEP_DELAY_NEW) OVER(
        PARTITION BY OP_UNIQUE_CARRIER
    ) AS promedio_aerolinea

FROM airline_catalog.bronze.flights_bronze;